# Reproducing AI Scientist

## Input Context
They specify a concrete experiment.py file that I link to below.

In [ ]:
TASK_DESCRIPTION = """You are given the following file to work with, that trains a low-dimensional diffusion model on 4 different 2D datasets. The diffusion model is based on DDPM. Particularly interesting ideas would involve controllable generation, e.g. biased towards different modes of the data, or new encodings for low-dimensional data aside from sinusoidal encodings."""
# They define a code experiment for context. Major limitation, as writing such example code is time-consuming.
import os
from src.horizon_bench.Config import WORKSPACE_PATH
with open(os.path.join(WORKSPACE_PATH, 'src/ai_scientist', "example_experiment.py"), "r") as f:
    EXPERIMENT_CODE = f.read()



MAX_ROUNDS = 5
# can be empty list
STARTING_IDEAS = [
  {
    "Name": "learning_rate_schedule",
    "Title": "Adaptive Learning Rate Schedules: Comparing different learning rate schedules for diffusion models.",
    "Experiment": "In this experiment, we compare the performance of different learning rate schedules on diffusion model performance. We use the final estimated KL as the evaluation metric.",
    "Interestingness": 4,
    "Feasibility": 10,
    "Novelty": 3
  }
]
SYSTEM_PROMPT = """You are an ambitious AI PhD student who is looking to publish a paper that will contribute significantly to the field."""


## Idea generation
They do this with a simple text chat without agents or tools. 

In [ ]:
from src.ai_scientist.prompts.idea_prompt import user_idea_prompt
import json


user_prompt = user_idea_prompt.format(
    task_description=TASK_DESCRIPTION, 
    code=EXPERIMENT_CODE if EXPERIMENT_CODE else "",
    prev_ideas_string="\n\n".join(json.dumps(x) for x in STARTING_IDEAS), 
    num_reflections=MAX_ROUNDS
)


In [ ]:
from src.horizon_bench.openai_api import chat

response = chat(user_prompt=user_prompt, system_prompt=SYSTEM_PROMPT)
response

In [ ]:
from src.horizon_bench.markdown import string_to_markdown
string_to_markdown(response.content)

In [ ]:
from src.ai_scientist.extract_json import extract_json
json_output = extract_json(response.content)

## Idea refinement
Idea gets refined in up to 5 rounds.

In [ ]:
from src.ai_scientist.prompts.idea_refinement_prompt import idea_refinement_prompt
from src.ai_scientist.extract_json import extract_json

message_history = [response]

for current_round_number in range(2, MAX_ROUNDS + 1):
    user_prompt = idea_refinement_prompt.format(
        current_round=current_round_number,
        num_reflections=MAX_ROUNDS,
    )

    response = chat(user_prompt=user_prompt, system_prompt=SYSTEM_PROMPT, message_history=message_history)
    message_history.append(response)
    
    string_to_markdown(response.content)
    
    # They use a brittle regex-based JSON extraction method.
    # I could easily replace this with OpenAI's inbuilt output parser.
    json_output = extract_json(response.content)
    assert json_output is not None, "Failed to extract JSON from LLM output"

    if "I am done" in response.content:
        print(f"Idea generation finished after {current_round_number - 1} rounds of refinement.")
        break

json_output

## Idea generation results

In [ ]:
from src.ai_scientist.extract_json import json_to_markdown
str_idea = json_to_markdown(json_output)
from IPython.display import Markdown, display
display(Markdown(str_idea))

In [ ]:
from src.ai_scientist.llm_as_a_judge.llm_as_a_judge import llm_as_a_judge
llm_as_a_judge_result = llm_as_a_judge(f"""
Name: {json_output['Name']}
Title: {json_output['Title']}
Experiment: {json_output['Experiment']}
""")

## LLM-as-a-judge evaluation of results
Scores are done according to the NeurIPS review form.  
Results are very bad, as far as I can tell. 

In [ ]:
display(Markdown(json_to_markdown(dict(llm_as_a_judge_result))))